**#Duka Electronics -Half-year sales Jan-Jun 2026**

In [22]:
%%writefile generate_data.py
import numpy as np
import pandas as pd

# --- Engagement 1: Duka Electronics (seed 2026) ---
rng = np.random.default_rng(2026)
n = 1000
products = {
    "Phone": 15000,
    "Laptop": 65000,
    "Headphones": 2500,
    "Tablet": 28000,
    "Charger": 1200,
}

sales = pd.DataFrame({
    "order_id": range(1001, 1001 + n),
    "order_date": pd.to_datetime("2026-01-01")
    + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
    "branch": rng.choice(
        ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"],
        n,
        p=[0.4, 0.2, 0.15, 0.15, 0.1],
    ),
    "product": rng.choice(list(products), n),
    "quantity": rng.integers(1, 6, n),
    "channel": rng.choice(
        ["In-store", "Online", "Phone order"],
        n,
        p=[0.55, 0.35, 0.10],
    ),
})
sales["unit_price"] = sales["product"].map(products)
sales.to_csv("duka_sales.csv", index=False)

# --- Engagement 2: Mobile Money Agent Network (seed 44) ---
rng = np.random.default_rng(44)
n = 600
agents = pd.DataFrame({
    "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
    "county": rng.choice(["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20),
    "onboarded": pd.date_range("2024-01-01", periods=20, freq="MS").strftime("%Y-%m-%d"),
})
txns = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
    "agent_id": rng.choice(agents["agent_id"], n),
    "txn_type": rng.choice(
        ["Deposit", "withdrawal", "DEPOSIT ", "Withdrawal", "deposit"], n
    ),
    "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
    "txn_date": (
        pd.to_datetime("2026-03-01")
        + pd.to_timedelta(rng.integers(0, 60, n), unit="D")
    ).strftime("%Y-%m-%d"),
})
idx = rng.choice(n, 30, replace=False)
txns.loc[idx[:15], "amount"] = np.nan
txns.loc[idx[15:20], "amount"] = "N/A"
txns.loc[idx[20:25], "amount"] = "250000"
txns.loc[idx[25:], "txn_date"] = txns.loc[idx[25:], "txn_date"].str.replace("-", "/")
txns = pd.concat([txns, txns.sample(12, random_state=1)], ignore_index=True)
txns.loc[rng.choice(len(txns), 8, replace=False), "agent_id"] = "AG099"
agents.to_csv("agents.csv", index=False)
txns.to_csv("agent_transactions.csv", index=False)

print("Done: wrote generate_data.py and generated CSVs.")

Overwriting generate_data.py


In [23]:
import os

for fname in ["generate_data.py", "duka_sales.csv", "agents.csv", "agent_transactions.csv"]:
    print(f"{fname}: {os.path.exists(fname)}")

generate_data.py: True
duka_sales.csv: True
agents.csv: True
agent_transactions.csv: True


In [24]:
!python generate_data.py

Done: wrote generate_data.py and generated CSVs.


In [25]:
import os

for fname in ["generate_data.py", "duka_sales.csv", "agents.csv", "agent_transactions.csv"]:
    print(f"{fname}: {os.path.exists(fname)}")

generate_data.py: True
duka_sales.csv: True
agents.csv: True
agent_transactions.csv: True


In [26]:
import os
import numpy as np
import pandas as pd

os.makedirs("outputs", exist_ok=True)
os.makedirs("excel", exist_ok=True)

# --- TASK 1: Load & First-Five-Minutes Inspection ---
df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

print("Shape:", df.shape)
print("\nData Types & Non-Nulls:")
df.info()
print("\nMissing values per column:\n", df.isna().sum())
print("\nDate range:", df["order_date"].min(), "to", df["order_date"].max())
print("\nUnique counts:\n", df.nunique())
print("\nBranch proportions:\n", df["branch"].value_counts(normalize=True))
print("\nChannel proportions:\n", df["channel"].value_counts(normalize=True))
display(df.head())
display(df.describe())

Shape: (1000, 7)

Data Types & Non-Nulls:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[ns]
 2   branch      1000 non-null   object        
 3   product     1000 non-null   object        
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   object        
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 54.8+ KB

Missing values per column:
 order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64

Date range: 2026-01-01 00:00:00 to 2026-06-29 00:00:00

Unique counts:
 order_id      1000
order_date     179
branch           5
product          5
quantity         5
channel          3
unit_price

,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


,order_id,order_date,quantity,unit_price
count,1000.000000,1000,1000.000000,1000.000000
mean,1500.500000,2026-03-30 08:44:09.600000,3.012000,23918.300000
min,1001.000000,2026-01-01 00:00:00,1.000000,1200.000000
25%,1250.750000,2026-02-14 00:00:00,2.000000,2500.000000
50%,1500.500000,2026-03-29 00:00:00,3.000000,15000.000000
75%,1750.250000,2026-05-14 00:00:00,4.000000,28000.000000
max,2000.000000,2026-06-29 00:00:00,5.000000,65000.000000
std,288.819436,NaN,1.416991,23908.790546


In [27]:
# TASK 2:Create the columns revenue, month, weekday, is_weekend and price_band (Premium ≥ KES 28,000; Value otherwise).
df["revenue"] = df["quantity"] * df["unit_price"]
df["month"] = df["order_date"].dt.month_name()
df["weekday"] = df["order_date"].dt.day_name()
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5
df["price_band"] = np.where(df["unit_price"] >= 28000, "Premium", "Value")

display(df[["order_id", "order_date", "revenue", "month", "weekday", "is_weekend", "price_band"]].head())

,order_id,order_date,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,45000,June,Wednesday,False,Value
1,1002,2026-02-02,260000,February,Monday,False,Premium
2,1003,2026-01-05,140000,January,Monday,False,Premium
3,1004,2026-04-26,12500,April,Sunday,True,Value
4,1005,2026-03-07,7500,March,Saturday,True,Value


In [28]:
# --- TASK 3: Branch Ranking by Revenue vs. Order Count ---
branch_summary = (
    df.groupby("branch", as_index=False)
    .agg(
        total_revenue=("revenue", "sum"),
        order_count=("order_id", "count"),
        avg_order_value=("revenue", "mean"),
    )
)
branch_summary["revenue_rank"] = branch_summary["total_revenue"].rank(ascending=False).astype(int)
branch_summary["orders_rank"] = branch_summary["order_count"].rank(ascending=False).astype(int)
branch_summary["revenue_share_pct"] = (branch_summary["total_revenue"] / branch_summary["total_revenue"].sum() * 100).round(2)
branch_summary = branch_summary.sort_values("total_revenue", ascending=False)
display(branch_summary)

,branch,total_revenue,order_count,avg_order_value,revenue_rank,orders_rank,revenue_share_pct
3,Nairobi,26106100,376,69431.117021,1,1,36.36
2,Mombasa,14003700,186,75288.709677,2,2,19.51
4,Nakuru,11970400,162,73891.358025,3,3,16.67
1,Kisumu,10868100,159,68352.830189,4,4,15.14
0,Eldoret,8844200,117,75591.452991,5,5,12.32


### Task 3: Branch Rankings by Total Revenue vs. Number of Orders

* Do the rankings match?** No, the rankings differ slightly

* **Top & bottom branches:** **Nairobi** leads both total revenue and order count (as expected from its 40% sampling share), , while Eldoret sits at the bottom.
* **Why revenue and order count can diverge:** Order count treats a single **Charger** (KES 1,200) or **Headphones** (KES 2,500) order the same as a multi-unit **Laptop** (KES 65,000) or **Tablet** (KES 28,000) order. A branch with slightly fewer orders but a higher average order value—or more high-ticket items and larger quantities—can earn more revenue than a branch with more small orders.


In [29]:
# --- TASK 4: Top 3 Products by Revenue vs. Units Sold ---
product_summary = (
    df.groupby("product", as_index=False)
    .agg(
        unit_price=("unit_price", "first"),
        total_revenue=("revenue", "sum"),
        units_sold=("quantity", "sum"),
        order_count=("order_id", "count"),
    )
)
print("Top 3 by Revenue:")
display(product_summary.sort_values("total_revenue", ascending=False).head(3))

print("Top 3 by Units Sold:")
display(product_summary.sort_values("units_sold", ascending=False).head(3))

Top 3 by Revenue:


,product,unit_price,total_revenue,units_sold,order_count
2,Laptop,65000,41535000,639,220
4,Tablet,28000,19348000,691,215
3,Phone,15000,8880000,592,194


Top 3 by Units Sold:


,product,unit_price,total_revenue,units_sold,order_count
4,Tablet,28000,19348000,691,215
2,Laptop,65000,41535000,639,220
3,Phone,15000,8880000,592,194


TASK 4: Top 3 Products by Revenue
vs. Units Sold ---



*  Laptops at KES 65,000 and Tablets at KES 28,000 generate huge revenue per unit sold, whereas lower-priced accessories like Chargers at KES 1,200 or Headphones at KES 2,500 can rank in the top three by volume without topping revenue



In [30]:
# --- TASK 5: Weekend vs. Weekday Comparison ---
weekend_summary = (
    df.groupby("is_weekend", as_index=False)
    .agg(
        order_count=("order_id", "count"),
        total_revenue=("revenue", "sum"),
        avg_order_revenue=("revenue", "mean"),
    )
)
weekend_summary["day_type"] = weekend_summary["is_weekend"].map({False: "Weekday (Mon-Fri)", True: "Weekend (Sat-Sun)"})
display(weekend_summary[["day_type", "order_count", "total_revenue", "avg_order_revenue"]])

,day_type,order_count,total_revenue,avg_order_revenue
0,Weekday (Mon-Fri),723,51856200,71723.651452
1,Weekend (Sat-Sun),277,19936300,71972.202166


TASK 5: Weekend vs. Weekday Comparison


* Weekdays have more more order
counts hence more revenue as compared to Weekend

In [31]:
# --- TASK 6: Best vs. Worst Month for Revenue ---
monthly_rev = (
    df.groupby(df["order_date"].dt.to_period("M"))["revenue"]
    .sum()
    .reset_index()
)
monthly_rev["month_name"] = monthly_rev["order_date"].dt.strftime("%B %Y")
display(monthly_rev.sort_values("revenue", ascending=False))

best_row = monthly_rev.loc[monthly_rev["revenue"].idxmax()]
worst_row = monthly_rev.loc[monthly_rev["revenue"].idxmin()]
pct_diff = ((best_row["revenue"] - worst_row["revenue"]) / worst_row["revenue"]) * 100

print(f"Best month: {best_row['month_name']} (KES {best_row['revenue']:,.0f})")
print(f"Worst month: {worst_row['month_name']} (KES {worst_row['revenue']:,.0f})")
print(f"Percentage difference (best vs worst): {pct_diff:.2f}%")

,order_date,revenue,month_name
3,2026-04,13518300,April 2026
2,2026-03,12396800,March 2026
0,2026-01,11786600,January 2026
1,2026-02,11772000,February 2026
4,2026-05,11415400,May 2026
5,2026-06,10903400,June 2026


Best month: April 2026 (KES 13,518,300)
Worst month: June 2026 (KES 10,903,400)
Percentage difference (best vs worst): 23.98%


TASK 6: Best vs. Worst Month for revenue


*   The month with the highest revenue is April while the month with the lowest revenue is June with a percentage differenceof 23.98%



In [32]:
# --- TASK 7: Premium Online Orders in Nairobi & Mombasa ---
prem_online_cc = df[
    (df["price_band"] == "Premium")
    & (df["channel"] == "Online")
    & (df["branch"].isin(["Nairobi", "Mombasa"]))
].copy()

print("Matching order count:", len(prem_online_cc))
print(f"Total revenue: KES {prem_online_cc['revenue'].sum():,.0f}")

prem_online_cc.to_csv("outputs/premium_online_coast_capital.csv", index=False)
print("Saved outputs/premium_online_coast_capital.csv")

Matching order count: 83
Total revenue: KES 10,218,000
Saved outputs/premium_online_coast_capital.csv


TASK 7: Premium Online Orders in Nairobi & Mombasa


*   The premium online orders were 83 with a total revenue of kes 10,218,000



In [33]:
# --- TASK 8: Pivot Table Reference to Verify Excel Totals ---
branch_channel_pivot = df.pivot_table(
    index="branch",
    columns="channel",
    values="revenue",
    aggfunc="sum",
    margins=True,
    margins_name="Total",
)
display(branch_channel_pivot)

channel,In-store,Online,Phone order,Total
branch,,,,
Eldoret,4885300,3450700,508200,8844200
Kisumu,6468600,3370800,1028700,10868100
Mombasa,8039600,4086800,1877300,14003700
Nairobi,14808400,7527400,3770300,26106100
Nakuru,6767300,4350400,852700,11970400
Total,40969200,22786100,8037200,71792500


## Executive Summary

* Across the 1,000 orders analyzed from January to June 2026, Duka Electronics generated KES 71,792,500 in total revenue[cite: 1].
* Nairobi leads all branches with [Nairobi revenue] in revenue across [Nairobi orders] orders, while [lowest branch] recorded the lowest revenue at [lowest branch revenue].
* [Top revenue product] leads total sales revenue at [top product revenue], whereas [top volume product] ranks highest by units sold with [top product units] units.
* Weekend orders generated [weekend revenue] across [weekend orders] transactions compared to [weekday revenue] across [weekday orders] weekday orders.
* [Best month] delivered the highest monthly revenue at [best month revenue] versus [worst month] at [worst month revenue] ([percentage difference]% difference), while Premium online orders from Nairobi and Mombasa totaled [filtered count] orders and [filtered revenue].